# GCE Pipeline 진단 노트북

**목적**: v7 결과에서 Pipeline / Zenodo ratio가 0.755로 나오는 원인을 추적.

**전제**: NFW² template normalization은 fit이 자유 c_gce를 갖기 때문에 SED에 영향을 주지 않음.
그러므로 0.75 차이는 다음 중 하나:
  - (A) Fit이 c_gce를 systematic하게 작게 잡고 있음 (gas/ics가 너무 큼)
  - (B) SED 재구성 단계의 unit/scale 문제
  - (C) Disk mask 정의 차이
  - (D) CCUBE counts 자체의 문제
  - (E) Exposure cube 자체의 문제
  - (F) gtmodel output의 정상성

**사용법**: v7 노트북을 끝까지 실행한 후, 같은 커널에서 이 셀을 차례대로 실행.

In [1]:
# %% ══════════════════════════════════════════════════════════
# 진단 셀 1: Fit 결과 진단
# Best-fit normalization parameters [c_gas, c_ics, c_gce, c_bub, c_iso]
# ══════════════════════════════════════════════════════════════

import numpy as np
import os
import pickle
from astropy.io import fits

# v7 결과 로드 (Cell 12에서 이미 로드되어 있다면 재사용)
try:
    _ = valid_results
    print('valid_results 메모리에 있음, 재사용')
except NameError:
    pkl_path = os.path.join(WORK_DIR, 'GCE_results_v7.pkl')
    with open(pkl_path, 'rb') as f:
        all_results = pickle.load(f)
    valid_results = [r for r in all_results if 'error' not in r]
    print(f'GCE_results_v7.pkl 로드 완료: {len(valid_results)}개 모델')

# Model X 결과 추출
model_x = next(r for r in valid_results if r['model'] == 'X')
norms = np.asarray(model_x['norms'])  # shape (14, 5)
errs = np.asarray(model_x['errors'])

print('\n=== Model X best-fit normalization parameters ===')
print(f"{'ie':>3} {'E[GeV]':>9} {'c_gas':>9} {'c_ics':>9} {'c_gce':>9} {'c_bub':>9} {'c_iso':>9}")
print('-' * 65)
for ie in range(N_ENERGY_BINS):
    print(f'{ie:3d} {ENERGY_CENTERS_GEV[ie]:9.3f} '
          f'{norms[ie,0]:9.3f} {norms[ie,1]:9.3f} '
          f'{norms[ie,2]:9.3f} {norms[ie,3]:9.3f} {norms[ie,4]:9.3f}')

print('\n=== 핵심 진단 ===')
print(f'c_gas 평균: {np.mean(norms[:,0]):.3f}  (정상이면 ~1.0)')
print(f'c_ics 평균: {np.mean(norms[:,1]):.3f}  (정상이면 ~1.0)')
print(f'c_gce 평균: {np.mean(norms[:,2]):.3f}  (정상이면 ~1.0, 작으면 GCE flux 작아짐)')
print(f'c_bub 평균: {np.mean(norms[:,3]):.3f}')
print(f'c_iso 평균: {np.mean(norms[:,4]):.3f}')
print()
print('해석:')
print('  - c_gce ≈ 1.0이면: NFW² template scale은 정상, 0.75는 다른 곳에')
print('  - c_gce ≈ 0.75이면: NFW² template이 데이터의 GCE보다 1.33배 큰 normalization을 가짐')
print('  - c_gas > 1.2: gas 모델이 GCE 영역의 flux를 흡수 중 (degeneracy)')

NameError: name 'WORK_DIR' is not defined

In [ ]:
# %% ══════════════════════════════════════════════════════════
# 진단 셀 2: 각 컴포넌트의 base 값 (raw, c와 무관)
# ══════════════════════════════════════════════════════════════

# disk-masked ROI 평균 base 값
S = OFFSET
E = S + NX_FIT

exp_crop = exp_cube_sr_full[:, S:E, S:E] if 'exp_cube_sr_full' in dir() else None
if exp_crop is None:
    # v7 노트북의 변수가 메모리에 없으면 직접 계산
    sr_per_pixel_loc = np.load(os.path.join(WORK_DIR, 'sr_per_pixel.npy'))
    exp_cube_data_loc = fits.open(expcube_center)[0].data
    exp_crop = (exp_cube_data_loc * sr_per_pixel_loc[np.newaxis, :, :])[:, S:E, S:E]

disk_mask_crop_loc = disk_mask_2d[S:E, S:E] if 'disk_mask_2d' in dir() else np.load(os.path.join(WORK_DIR, 'disk_mask.npy'))[S:E, S:E]
ds = disk_mask_crop_loc.sum()
print(f'disk_mask_crop 픽셀 수: {ds:.0f} (40x40 ROI 중 {100*ds/(NX_FIT*NY_FIT):.1f}%)')

model_name = 'X'
components = ['pion', 'bremss', 'ics', 'GCE', 'Fermi_bubble', 'isotropic']

print(f"\n=== Non-convol component map base values (Model X) ===")
print(f"base[ie] = sum(disk_mask × cmap_nc[ie] / exp_sr[ie]) / sum(disk_mask)")
print(f"단위: 1/(GeV cm² s sr) 비슷 (실제로는 1/cm²/s/sr in counts/exp scale)")
print()
header = f"{'ie':>3} {'E[GeV]':>8}"
for c in components:
    header += f" {c[:8]:>10}"
print(header)
print('-' * 80)

base_table = np.zeros((N_ENERGY_BINS, len(components)))
for ic, comp in enumerate(components):
    fpath = os.path.join(COMPMAP_DIR_NC, f'{comp}_model{model_name}.fits')
    if not os.path.exists(fpath):
        print(f'⚠️ {comp} 파일 없음: {fpath}')
        continue
    with fits.open(fpath) as h:
        cmap = h[0].data[:, S:E, S:E]
    for ie in range(N_ENERGY_BINS):
        base_table[ie, ic] = np.sum(disk_mask_crop_loc * cmap[ie] / np.maximum(exp_crop[ie], 1e-30)) / ds

for ie in range(N_ENERGY_BINS):
    row = f'{ie:3d} {ENERGY_CENTERS_GEV[ie]:8.3f}'
    for ic in range(len(components)):
        row += f' {base_table[ie,ic]:10.3e}'
    print(row)

print()
print('=== Sanity check (Model X, ie=6 ~1.5 GeV) ===')
ie6 = 6
print(f'  E = {ENERGY_CENTERS_GEV[ie6]:.3f} GeV')
print(f'  ENERGY_WIDTHS_GEV[6] = {ENERGY_WIDTHS_GEV[ie6]:.4f} GeV')
print(f'  E²/dE = {ENERGY_CENTERS_GEV[ie6]**2/ENERGY_WIDTHS_GEV[ie6]:.4f} GeV')
print()
print(f'  GCE base[6]   = {base_table[ie6,3]:.3e}  (1/cm²/s/sr 비슷)')
print(f'  c_gce[6]      = {norms[ie6,2]:.3f}')
print(f'  → SED contribution = c_gce × base × E²/dE')
print(f'                     = {norms[ie6,2]} × {base_table[ie6,3]:.3e} × {ENERGY_CENTERS_GEV[ie6]**2/ENERGY_WIDTHS_GEV[ie6]:.3f}')
print(f'                     = {norms[ie6,2] * base_table[ie6,3] * ENERGY_CENTERS_GEV[ie6]**2 / ENERGY_WIDTHS_GEV[ie6]:.3e} GeV/cm²/s/sr')
print(f'  Zenodo @ 1.5 GeV ≈ ~9.2e-7 GeV/cm²/s/sr')

In [ ]:
# %% ══════════════════════════════════════════════════════════
# 진단 셀 3: CCUBE 데이터, exposure 자체의 정상성 확인
# ══════════════════════════════════════════════════════════════

print('=== CCUBE counts data (raw) ===')
with fits.open(ccube_file) as h:
    counts = h[0].data

print(f'CCUBE shape: {counts.shape}')
print(f'전체 counts (all 600x600, all 14 bins): {counts.sum():.3e}')
print(f'40x40 ROI counts: {counts[:, S:E, S:E].sum():.3e}')
print()
print(f"{'ie':>3} {'E[GeV]':>8} {'counts(40x40)':>14} {'counts/pix':>12}")
print('-' * 50)
for ie in range(N_ENERGY_BINS):
    c_ie = counts[ie, S:E, S:E]
    print(f'{ie:3d} {ENERGY_CENTERS_GEV[ie]:8.3f} {c_ie.sum():14.0f} {c_ie.mean():12.4f}')

print()
print('=== Exposure cube ===')
print(f'exp_cube[6, 200, 200] = {exp_cube_data_loc[6, 200, 200] if "exp_cube_data_loc" in dir() else fits.open(expcube_center)[0].data[6, 200, 200]:.3e} cm² s')
print(f'exp_cube_sr[6, 200, 200] = {exp_crop[6, 200, 200]:.3e} cm² s sr')
print(f'sr_per_pixel @ center: {(0.1*np.pi/180)**2:.3e} sr (flat) — 우리 코드는 cos(b)를 곱한 값 사용')
print()
print('기대값: 12.5 yr × 0.5 m² × ~0.1 sr ~ 1e10 cm²·s 정도가 정상')

In [ ]:
# %% ══════════════════════════════════════════════════════════
# 진단 셀 4: Sanghwan 공식과 우리 공식 직접 비교 (1 bin)
# ══════════════════════════════════════════════════════════════
# Sanghwan SED 공식 (line 870-874, 1007):
#   GCE_base[i] = sum(disk_mask × cmap[i] / exp_cube[i]) / sum(disk_mask)
#                 (여기서 exp_cube = exposure × steradian_per_pixel)
#   SED = E[i]² × GCE_base × c_gce / delta_E[i]
#
# 우리 Cell 12 공식:
#   동일 (compute_component_sed)
#
# 차이가 있다면 ENERGY_WIDTHS_GEV (= delta_E)와 sr 처리에서 발생

print('=== ENERGY_EDGES와 ENERGY_WIDTHS 검증 ===')
print(f'우리 ENERGY_EDGES_GEV (Table III에서):')
for i, e in enumerate(ENERGY_EDGES_GEV):
    print(f'  edge[{i}] = {e:.4f} GeV')

print(f'\nENERGY_WIDTHS_GEV (= edges[1:] - edges[:-1]):')
for ie in range(N_ENERGY_BINS):
    print(f'  bin[{ie}] center={ENERGY_CENTERS_GEV[ie]:.4f}, width={ENERGY_WIDTHS_GEV[ie]:.4f} GeV')

# CCUBE의 EBOUNDS HDU 직접 비교
print('\n=== CCUBE EBOUNDS HDU 값 (Sanghwan이 직접 읽는 것) ===')
with fits.open(ccube_file) as h:
    if 'EBOUNDS' in [hdu.name for hdu in h]:
        ebounds = h['EBOUNDS'].data
        print(f'EBOUNDS HDU 발견, shape={len(ebounds)}')
        print(f'columns: {ebounds.dtype.names}')
        for i, row in enumerate(ebounds):
            E_min_keV = row[1]  # E_MIN column (Fermi convention: keV)
            E_max_keV = row[2]
            E_geom_GeV = np.sqrt(E_min_keV * E_max_keV) * 1e-6
            dE_GeV = (E_max_keV - E_min_keV) * 1e-6
            print(f'  bin[{i}]: E_min={E_min_keV*1e-6:.4f} GeV, E_max={E_max_keV*1e-6:.4f} GeV, '
                  f'E_geom={E_geom_GeV:.4f}, dE={dE_GeV:.4f}')
    else:
        print('⚠️ EBOUNDS HDU 없음')
        # 다른 HDU 시도
        for hdu in h:
            print(f'  HDU: {hdu.name}, type={type(hdu).__name__}')

print('\n=== 결정적 비교 ===')
print('만약 CCUBE의 EBOUNDS와 우리 ENERGY_EDGES가 다르면 → 그게 0.75의 원인')
print('특히 dE 비율이 0.75라면 → SED scale 차이 = 1.333 → ratio 0.75 해명')

In [ ]:
# %% ══════════════════════════════════════════════════════════
# 진단 셀 5: Zenodo 파일 직접 조사
# ══════════════════════════════════════════════════════════════

zenodo_file = '/home/haebarg/GCE-Chi-square-fitting/GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra/GCE_ModelX_flux_Inner40x40_masked_disk.dat'

print('=== Zenodo Model X 파일 정밀 분석 ===')
print(f'경로: {zenodo_file}')
print()

# Header 읽기
with open(zenodo_file, 'r') as f:
    head_lines = []
    for line in f:
        if line.startswith('#') or not any(c.isdigit() for c in line[:5]):
            head_lines.append(line.rstrip())
        else:
            break

if head_lines:
    print('헤더 라인들:')
    for h in head_lines:
        print(f'  {h}')
else:
    print('헤더 없음 — 그냥 숫자 데이터')

g = np.loadtxt(zenodo_file)
print(f'\n전체 shape: {g.shape}')
print(f'\n첫 5 row 직접 출력:')
for i in range(min(5, len(g))):
    print(f'  row {i}: {g[i]}')

print(f'\n=== 컬럼 의미 추측 ===')
print(f'col 0 (energy): {g[:, 0]}')
print(f'col 1 (flux?): {g[:, 1]}')
if g.shape[1] > 2:
    print(f'col 2 (err lo?): {g[:, 2]}')
if g.shape[1] > 3:
    print(f'col 3 (err hi?): {g[:, 3]}')

# col 0의 단위 추측
print(f'\ncol 0 range: [{g[:, 0].min():.3e}, {g[:, 0].max():.3e}]')
if g[:, 0].max() < 100:
    print('  → GeV 단위로 추측')
elif g[:, 0].max() < 1e5:
    print('  → MeV 단위로 추측')

# col 1 range
print(f'\ncol 1 range: [{g[:, 1].min():.3e}, {g[:, 1].max():.3e}]')
print('  GeV/cm²/s/sr 단위 (Cholis Fig 12)이면 ~1e-7 ~ 1e-6 정도가 정상')

# 우리 ENERGY_CENTERS_GEV와 일치하는지
print(f'\n=== Energy bin 일치 검증 ===')
print(f'{"Zenodo E":>10} {"우리 E":>10} {"비율":>10}')
for ie in range(min(len(g), N_ENERGY_BINS)):
    r = g[ie, 0] / ENERGY_CENTERS_GEV[ie]
    print(f'{g[ie,0]:10.4f} {ENERGY_CENTERS_GEV[ie]:10.4f} {r:10.4f}')
print('\n비율이 1.0이 아니면 단위 차이 (예: GeV vs MeV)')

In [ ]:
# %% ══════════════════════════════════════════════════════════
# 진단 셀 6: 종합 진단
# ══════════════════════════════════════════════════════════════

print('=' * 70)
print('GCE Pipeline 진단 종합')
print('=' * 70)

ratio_avg = np.mean(norms[:, 2])
print(f'\n[A] FIT 단계: c_gce 평균 = {ratio_avg:.3f}')
if abs(ratio_avg - 1.0) < 0.1:
    print('    → c_gce가 1에 가까움 = NFW² template scale은 정상')
    print('    → 0.75 ratio는 SED 재구성 이후 단계의 문제')
    print('    → 다음 단계: 진단 셀 4 (energy bin 정의)와 셀 5 (Zenodo 파일) 검토')
elif ratio_avg < 0.85:
    print('    → c_gce가 작음 = NFW² template normalization이 데이터보다 큼')
    print('    → 가능한 원인:')
    print('       1. 우리 NFW² 계산의 absolute scale이 잘못됨')
    print('       2. PSF가 데이터에 적용되어 GCE를 spread out 시킴')
    print('       3. gas/ics가 GCE 영역의 flux를 흡수 (degeneracy)')
    c_gas_avg = np.mean(norms[:, 0])
    c_ics_avg = np.mean(norms[:, 1])
    print(f'    → c_gas 평균 = {c_gas_avg:.3f}, c_ics 평균 = {c_ics_avg:.3f}')
    if c_gas_avg > 1.1 or c_ics_avg > 1.1:
        print('       ⚠️ gas 또는 ics가 1.0 초과 → degeneracy 의심')
elif ratio_avg > 1.15:
    print('    → c_gce가 큼 = 데이터가 template보다 더 많은 GCE를 가짐')
    print('    → SED는 커지고 ratio도 1보다 커야 하는데 0.755 ← 모순')
    print('    → SED 재구성 단계에서 작아지는 cancellation이 있음')

print('\n다음 진단 우선순위:')
print('  1. 진단 셀 1, 2: c_gce 값과 base 값을 보고 원인 분류')
print('  2. 진단 셀 4: ENERGY_WIDTHS가 CCUBE와 일치하는지')
print('  3. 진단 셀 5: Zenodo 파일이 우리가 가정한 단위/컬럼인지')